# RetailPulse: AI-Powered Customer Analytics & Demand Forecasting

## Notebook 4: Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "../data/processed/online_retail_II_cleaned.csv"

df = pd.read_csv(file_path)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.shape)
print(df.columns)

(779425, 8)
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')


### Convert Invoice Date

Convert the `InvoiceDate` column into datetime format so that date and time features can be extracted.

In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceDate"].dtype

dtype('<M8[us]')

### Extract Date Features

Create new features from the `InvoiceDate` column to support time-based analysis and demand forecasting.

In [5]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["MonthName"] = df["InvoiceDate"].dt.month_name()
df["Quarter"] = df["InvoiceDate"].dt.quarter
df["Day"] = df["InvoiceDate"].dt.day
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

In [6]:
df[[
    "InvoiceDate",
    "Year",
    "Month",
    "MonthName",
    "Quarter",
    "Day",
    "DayOfWeek",
    "Hour"
]].head()

,InvoiceDate,Year,Month,MonthName,Quarter,Day,DayOfWeek,Hour
0,2009-12-01 07:45:00,2009,12,December,4,1,Tuesday,7
1,2009-12-01 07:45:00,2009,12,December,4,1,Tuesday,7
2,2009-12-01 07:45:00,2009,12,December,4,1,Tuesday,7
3,2009-12-01 07:45:00,2009,12,December,4,1,Tuesday,7
4,2009-12-01 07:45:00,2009,12,December,4,1,Tuesday,7


### Create Revenue Feature

Calculate the revenue for each transaction by multiplying the quantity of items purchased by their unit price.

In [7]:
df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [8]:
df["Revenue"].describe()

count    779425.000000
mean         22.291823
std         227.427075
min           0.001000
25%           4.950000
50%          12.480000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64

### Create Reference Date

Use the latest transaction date in the dataset to calculate customer recency.

In [9]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Reference Date:", reference_date)

Reference Date: 2011-12-10 12:50:00


### Calculate RFM Features

Create Recency, Frequency, and Monetary features for each customer.

In [10]:
rfm = (
    df.groupby("Customer ID")
      .agg({
          "InvoiceDate": lambda x: (reference_date - x.max()).days,
          "Invoice": "nunique",
          "Revenue": "sum"
      })
      .reset_index()
)

rfm.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


### RFM Summary Statistics

Analyze the distribution of the Recency, Frequency, and Monetary features.

In [11]:
rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


### Check Missing Values

Verify that the RFM dataset does not contain any missing values.

In [12]:
rfm.isnull().sum()

CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64

In [13]:
df.to_csv("../data/processed/online_retail_II_feature_engineered.csv", index=False)

In [14]:
rfm.to_csv("../data/processed/customer_rfm.csv", index=False)

### Create Time of Day Feature

Categorize each transaction into Morning, Afternoon, Evening, or Night based on the transaction hour.

In [15]:
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["TimeOfDay"] = df["Hour"].apply(get_time_of_day)

df[["Hour", "TimeOfDay"]].head()

,Hour,TimeOfDay
0,7,Morning
1,7,Morning
2,7,Morning
3,7,Morning
4,7,Morning


### Create Season Feature

Assign each transaction to a season based on the transaction month.

In [16]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["Season"] = df["Month"].apply(get_season)

df[["Month", "Season"]].head()

,Month,Season
0,12,Winter
1,12,Winter
2,12,Winter
3,12,Winter
4,12,Winter


### Create Basket Size Feature

Calculate the total number of items purchased in each invoice.

In [17]:
basket_size = (
    df.groupby("Invoice")["Quantity"]
      .sum()
      .reset_index()
)

basket_size.columns = ["Invoice", "BasketSize"]

df = df.merge(basket_size, on="Invoice")

df[["Invoice", "BasketSize"]].head()

,Invoice,BasketSize
0,489434,166
1,489434,166
2,489434,166
3,489434,166
4,489434,166


### Create Unique Products Feature

Calculate the number of unique products purchased in each invoice.

In [18]:
unique_products = (
    df.groupby("Invoice")["StockCode"]
      .nunique()
      .reset_index()
)

unique_products.columns = ["Invoice", "UniqueProducts"]

df = df.merge(unique_products, on="Invoice")

df[["Invoice", "UniqueProducts"]].head()

,Invoice,UniqueProducts
0,489434,8
1,489434,8
2,489434,8
3,489434,8
4,489434,8


In [19]:
df.to_csv("../data/processed/online_retail_II_feature_engineered.csv", index=False)

In [20]:
df[["TimeOfDay", "Season", "BasketSize", "UniqueProducts"]].head()

,TimeOfDay,Season,BasketSize,UniqueProducts
0,Morning,Winter,166,8
1,Morning,Winter,166,8
2,Morning,Winter,166,8
3,Morning,Winter,166,8
4,Morning,Winter,166,8


In [21]:
df[["TimeOfDay", "Season", "BasketSize", "UniqueProducts"]].head(10)

,TimeOfDay,Season,BasketSize,UniqueProducts
0,Morning,Winter,166,8
1,Morning,Winter,166,8
2,Morning,Winter,166,8
3,Morning,Winter,166,8
4,Morning,Winter,166,8
5,Morning,Winter,166,8
6,Morning,Winter,166,8
7,Morning,Winter,166,8
8,Morning,Winter,60,4
9,Morning,Winter,60,4


In [22]:
df = pd.read_csv("../data/processed/online_retail_II_feature_engineered.csv")

print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Year', 'Month', 'MonthName', 'Quarter', 'Day', 'DayOfWeek', 'Hour', 'Revenue', 'TimeOfDay', 'Season', 'BasketSize', 'UniqueProducts']
